# Évaluer un classifieur : prédire le départ de clients

Un opérateur télécom a perdu, le mois dernier, plus d'un quart de ses clients. Il voudrait repérer à l'avance ceux qui risquent de partir (le *churn*) pour leur proposer une offre de fidélisation. Un modèle qui prédit ces départs ne sert que si l'on sait l'évaluer : c'est l'objet de ce TP.

Le TP a deux parties (environ 1 h 15 pour les deux) :

1. **Évaluer un classifieur** (exercices 1 à 4) : une référence naïve, une régression logistique, sa matrice de confusion, sa précision et son rappel, puis la validation croisée. Une formation d'initiation peut s'arrêter là.
2. **Comparer deux modèles et choisir un seuil** (exercices 5 à 7) : un classifieur bayésien naïf face à la régression logistique, leurs courbes ROC et leur AUC, puis le seuil de décision choisi d'après le coût des erreurs.

La préparation des données est donnée toute faite : les exercices portent sur l'évaluation.

Données : [Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d) (IBM, licence Apache 2.0), 7 043 clients fictifs.

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
  ConfusionMatrixDisplay,
  RocCurveDisplay,
  accuracy_score,
  classification_report,
  precision_score,
  recall_score,
  roc_auc_score,
)
from sklearn.model_selection import (
  StratifiedKFold,
  cross_val_predict,
  cross_validate,
  train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Les données

Chaque ligne est un client : son contrat, ses services, ses factures, et la cible, `Parti`, qui vaut 1 s'il est parti le mois dernier et 0 sinon.

In [ ]:
clients = pd.read_csv(
  "https://raw.githubusercontent.com/shuuchuu/datasets/main/telco-churn/telco-churn.csv"
)

# Noms de colonnes et valeurs en français
colonnes = {
  "gender": "Genre",
  "SeniorCitizen": "Senior",
  "Partner": "En couple",
  "Dependents": "Personnes à charge",
  "tenure": "Ancienneté (mois)",
  "PhoneService": "Téléphone",
  "MultipleLines": "Plusieurs lignes",
  "InternetService": "Internet",
  "OnlineSecurity": "Sécurité en ligne",
  "OnlineBackup": "Sauvegarde en ligne",
  "DeviceProtection": "Protection des appareils",
  "TechSupport": "Support technique",
  "StreamingTV": "TV en streaming",
  "StreamingMovies": "Films en streaming",
  "Contract": "Contrat",
  "PaperlessBilling": "Facture dématérialisée",
  "PaymentMethod": "Moyen de paiement",
  "MonthlyCharges": "Facture mensuelle ($)",
  "TotalCharges": "Total facturé ($)",
  "Churn": "Parti",
}
valeurs = {
  "Yes": "Oui",
  "No": "Non",
  "Male": "Homme",
  "Female": "Femme",
  "No phone service": "Pas de téléphone",
  "No internet service": "Pas d'internet",
  "Fiber optic": "Fibre",
  "DSL": "ADSL",
  "Month-to-month": "Sans engagement",
  "One year": "1 an",
  "Two year": "2 ans",
  "Electronic check": "Chèque électronique",
  "Mailed check": "Chèque postal",
  "Bank transfer (automatic)": "Virement automatique",
  "Credit card (automatic)": "Carte bancaire automatique",
}

clients = clients.drop(columns="customerID").rename(columns=colonnes).replace(valeurs)
clients["Senior"] = clients["Senior"].map({0: "Non", 1: "Oui"})
# Les nouveaux clients n'ont encore rien payé : leur total est vide
clients["Total facturé ($)"] = pd.to_numeric(
  clients["Total facturé ($)"], errors="coerce"
).fillna(0)
clients["Parti"] = (clients["Parti"] == "Oui").astype(int)

print(f"{len(clients)} clients, dont {clients['Parti'].mean():.1%} sont partis")
clients.head()

On met de côté un quart des clients pour le **test**, la mesure finale : aucun choix ne se fera sur eux. La séparation est stratifiée (`stratify=y`) : le taux de départ est le même dans l'entraînement et dans le test.

In [ ]:
X = clients.drop(columns="Parti")
y = clients["Parti"]
X_entrainement, X_test, y_entrainement, y_test = train_test_split(
  X, y, test_size=0.25, stratify=y, random_state=0
)
print(f"Entraînement : {len(X_entrainement)} clients, test : {len(X_test)} clients")
print(
  f"Taux de départ : {y_entrainement.mean():.1%} (entraînement), "
  f"{y_test.mean():.1%} (test)"
)

Le prétraitement, donné tout fait : les colonnes catégorielles sont encodées en one-hot (une colonne de 0 et de 1 par valeur ; `handle_unknown="ignore"` ignore une valeur que l'entraînement n'a pas vue), les trois colonnes numériques sont standardisées. Mis dans un pipeline avec le modèle, il ne s'ajuste que sur les données d'entraînement : pas de fuite du test vers le modèle.

In [ ]:
colonnes_numeriques = [
  "Ancienneté (mois)",
  "Facture mensuelle ($)",
  "Total facturé ($)",
]
colonnes_categorielles = [
  colonne for colonne in X.columns if colonne not in colonnes_numeriques
]

pretraitement = ColumnTransformer(
  [
    (
      "categorielles",
      OneHotEncoder(handle_unknown="ignore", sparse_output=False),
      colonnes_categorielles,
    ),
    ("numeriques", StandardScaler(), colonnes_numeriques),
  ]
)

nb_colonnes = pretraitement.fit_transform(X_entrainement).shape[1]
print(
  f"{len(colonnes_categorielles)} colonnes catégorielles, "
  f"{len(colonnes_numeriques)} numériques : {nb_colonnes} après le prétraitement"
)

## Partie 1 — Évaluer un classifieur

Une question guide cette partie : le modèle repère-t-il les clients qui vont partir ?

### 1. Une référence naïve

Avant tout modèle, une référence (*baseline*) : un classifieur qui ignore les données et prédit toujours la classe la plus fréquente, ici « reste ». Un modèle ne vaut que s'il fait mieux qu'elle.

*Créez une référence `reference` avec `DummyClassifier`, qui prédit toujours la classe majoritaire, entraînez-la sur l'entraînement, puis affichez son exactitude (accuracy) sur le test.*

**Documentation utile**

- [`sklearn.dummy.DummyClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html)
- [`sklearn.metrics.accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `DummyClassifier(strategy="most_frequent")` : prédit toujours la classe la plus fréquente de l'entraînement.
- `.fit(X_entrainement, y_entrainement)`, puis `.predict(X_test)` : les prédictions, à passer à `accuracy_score(y_test, predictions)`, qui renvoie une proportion entre 0 et 1.
- Exactitude attendue : la part des clients restés dans le test, 73,5 %.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
reference = DummyClassifier(strategy=...)
reference.fit(..., ...)
print(f"Exactitude de la référence : {accuracy_score(y_test, ...):.1%}")
```

#### Solution

In [ ]:
reference = DummyClassifier(strategy="most_frequent")
reference.fit(X_entrainement, y_entrainement)
print(
  f"Exactitude de la référence : {accuracy_score(y_test, reference.predict(X_test)):.1%}"
)

73,5 % de bonnes réponses sans rien apprendre : c'est la part des clients restés. Un score de 80 % paraît bon dans l'absolu ; face à cette référence, il l'est beaucoup moins. Et la référence ne repère aucun départ : l'exercice 3 le mesure.

### 2. Une régression logistique

*Créez un modèle `modele` qui enchaîne le prétraitement `pretraitement` et une `LogisticRegression` avec `make_pipeline`, entraînez-le sur l'entraînement, puis affichez son exactitude sur le test. Combien de points gagne-t-il sur la référence ?*

**Documentation utile**

- [`sklearn.linear_model.LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `make_pipeline(pretraitement, LogisticRegression(max_iter=1000))` : `max_iter=1000` laisse à l'optimisation assez d'itérations pour converger (le défaut, 100, peut ne pas suffire et déclencher un avertissement).
- Le pipeline s'utilise comme un modèle : `modele.fit(X_entrainement, y_entrainement)`, puis `modele.predict(X_test)`.
- Exactitude attendue : environ 79 %.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele = make_pipeline(pretraitement, LogisticRegression(max_iter=...))
modele.fit(..., ...)
print(f"Exactitude de la régression logistique : {accuracy_score(y_test, ...):.1%}")
```

#### Solution

In [ ]:
modele = make_pipeline(pretraitement, LogisticRegression(max_iter=1000))
modele.fit(X_entrainement, y_entrainement)
print(
  f"Exactitude de la régression logistique : "
  f"{accuracy_score(y_test, modele.predict(X_test)):.1%}"
)

79,2 % contre 73,5 % : moins de 6 points de mieux que la référence. Est-ce un bon modèle ? L'exactitude ne dit pas où sont ses erreurs.

### 3. Matrice de confusion, précision et rappel

*Tracez la matrice de confusion de `modele` sur le test avec `ConfusionMatrixDisplay.from_estimator`, puis affichez la précision, le rappel et le score F1 de chaque classe avec `classification_report`. Combien des clients partis du test le modèle laisse-t-il passer ? Calculez enfin le rappel de la référence avec `recall_score`.*

Dans la matrice, comme dans le cours, les lignes sont la réalité et les colonnes la prédiction ; la classe positive est 1, « parti ».

**Documentation utile**

- [`sklearn.metrics.ConfusionMatrixDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_estimator)
- [`sklearn.metrics.classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [`sklearn.metrics.recall_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `ConfusionMatrixDisplay.from_estimator(modele, X_test, y_test, display_labels=["Resté", "Parti"])` : `display_labels` nomme les classes 0 et 1 ; `plt.show()` affiche la figure.
- `classification_report(y_test, modele.predict(X_test), target_names=["Resté", "Parti"])` renvoie un texte, à passer à `print`. Sa ligne `Parti` donne la précision, le rappel et le F1 des départs.
- Les clients partis que le modèle laisse passer sont les faux négatifs : la case de la ligne « Parti » et de la colonne « Resté ».
- `recall_score(y_test, reference.predict(X_test))` : le rappel de la classe 1.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
ConfusionMatrixDisplay.from_estimator(
  modele, ..., ..., display_labels=["Resté", "Parti"]
)
plt.show()
print(classification_report(y_test, ..., target_names=["Resté", "Parti"]))
print(f"Rappel de la référence : {recall_score(y_test, ...):.1%}")
```

#### Solution

In [ ]:
ConfusionMatrixDisplay.from_estimator(
  modele, X_test, y_test, display_labels=["Resté", "Parti"]
)
plt.show()
print(
  classification_report(y_test, modele.predict(X_test), target_names=["Resté", "Parti"])
)
print(f"Rappel de la référence : {recall_score(y_test, reference.predict(X_test)):.1%}")

Sur les 467 clients partis du test, le modèle en repère 248 (un rappel de 53 %) et en laisse passer 219 : près d'un sur deux. Quand il annonce un départ, il a raison 63 fois sur 100 (une précision de 63 %). La référence, elle, ne repère aucun départ : son rappel est nul, malgré ses 73,5 % d'exactitude. Sur des classes déséquilibrées, l'exactitude trompe : regardez la précision et le rappel de la classe qui compte.

### 4. La validation croisée

Un seul découpage entre entraînement et test donne un score qui dépend des clients tirés dans le test. La validation croisée découpe l'entraînement en 5 plis, entraîne 5 modèles et donne la moyenne et l'écart type des 5 scores de validation : de quoi comparer des modèles sans toucher au test, gardé pour la mesure finale.

*Créez un découpage `plis` en 5 plis stratifiés et mélangés avec `StratifiedKFold`, puis écrivez une fonction `evaluer(classifieur, scores)` qui évalue `classifieur` par validation croisée sur l'entraînement avec `cross_validate`, pour chaque score de la liste `scores` (des noms de scores de scikit-learn, comme `"recall"`), et renvoie un `DataFrame` d'une ligne par score, avec la moyenne et l'écart type sur les 5 plis. La section qui suit compare la référence et la régression logistique.*

**Documentation utile**

- [`sklearn.model_selection.StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [`sklearn.model_selection.cross_validate`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html)
- [Les noms des scores de scikit-learn](https://scikit-learn.org/stable/modules/model_evaluation.html#string-name-scorers)

In [ ]:
def evaluer(classifieur, scores):
  # Votre code ici
  ...

#### Conseils supplémentaires

- `StratifiedKFold(n_splits=5, shuffle=True, random_state=0)` : 5 plis qui gardent chacun le taux de départ, tirés après un mélange reproductible. Créez `plis` avant la fonction : l'exercice 7 s'en sert aussi.
- `cross_validate(classifieur, X_entrainement, y_entrainement, cv=plis, scoring=scores)` renvoie un dictionnaire : pour chaque score `s`, la clé `f"test_{s}"` donne un tableau de 5 valeurs, une par pli.
- `.mean()` et `.std()` d'un tableau NumPy : sa moyenne et son écart type.
- `pd.DataFrame({"moyenne": [...], "écart type": [...]}, index=scores)` : une ligne par score.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
plis = StratifiedKFold(n_splits=5, shuffle=..., random_state=0)


def evaluer(classifieur, scores):
  resultats = cross_validate(
    classifieur, X_entrainement, y_entrainement, cv=..., scoring=...
  )
  return pd.DataFrame(
    {
      "moyenne": [resultats[f"test_{score}"].mean() for score in scores],
      "écart type": [... for score in scores],
    },
    index=scores,
  )
```

#### Solution

In [ ]:
plis = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def evaluer(classifieur, scores):
  resultats = cross_validate(
    classifieur, X_entrainement, y_entrainement, cv=plis, scoring=scores
  )
  return pd.DataFrame(
    {
      "moyenne": [resultats[f"test_{score}"].mean() for score in scores],
      "écart type": [resultats[f"test_{score}"].std() for score in scores],
    },
    index=scores,
  )

### La référence face à la régression logistique

`evaluer` compare les deux modèles de la partie 1, sur l'exactitude, la précision, le rappel et le F1 :

In [ ]:
scores_partie_1 = ["accuracy", "precision", "recall", "f1"]
pd.concat(
  {
    "Référence": evaluer(reference, scores_partie_1),
    "Régression logistique": evaluer(modele, scores_partie_1),
  },
  axis=1,
).round(3)

En validation croisée, la régression logistique obtient 80,7 % d'exactitude (± 2,2 points d'un pli à l'autre), une précision de 66 % (± 5) et un rappel de 55 % (± 3) ; la référence, 73,5 % d'exactitude et des zéros partout ailleurs (scikit-learn avertit que sa précision n'est pas définie, puisqu'elle ne prédit aucun départ, et la met à 0). L'écart type dit ce que vaut une différence : un écart entre deux modèles plus petit que ces variations d'un pli à l'autre ne suffit pas à les départager.

Ces scores ne sont pas ceux du test (79,2 % d'exactitude, 53 % de rappel) : ce sont d'autres clients, et l'écart reste dans la marge d'un pli à l'autre.

## Partie 2 — Comparer deux modèles et choisir un seuil

La régression logistique n'est pas le seul classifieur possible. Comment en comparer deux ? Et comment passer d'une probabilité de départ à une décision : faire une offre à ce client, ou non ?

Si votre formation s'arrête à la partie 1, passez à la conclusion.

### 5. Un second classifieur : le bayésien naïf

*Créez un modèle `bayes` qui enchaîne le même prétraitement et un `GaussianNB` (le classifieur bayésien naïf pour des variables numériques), puis comparez-le à `modele` avec `evaluer`, sur les scores `"roc_auc"` et `"recall"`. Lequel repère le plus de départs ? Lequel a la meilleure AUC ?*

L'AUC (`roc_auc`) est l'aire sous la courbe ROC, que l'exercice 6 trace : elle mesure si le modèle donne une probabilité de départ plus haute aux clients partis qu'aux autres, quel que soit le seuil.

**Documentation utile**

- [`sklearn.naive_bayes.GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `make_pipeline(pretraitement, GaussianNB())` : comme `modele`, avec un autre classifieur.
- `evaluer` entraîne lui-même un modèle par pli : inutile d'appeler `bayes.fit` ici.
- `pd.concat({"Régression logistique": ..., "Bayésien naïf": ...}, axis=1)` met les deux tableaux côte à côte, comme la cellule qui suit l'exercice 4.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
bayes = make_pipeline(pretraitement, ...)
scores_partie_2 = ["roc_auc", "recall"]
pd.concat(
  {
    "Régression logistique": evaluer(modele, scores_partie_2),
    "Bayésien naïf": evaluer(..., ...),
  },
  axis=1,
).round(3)
```

#### Solution

In [ ]:
bayes = make_pipeline(pretraitement, GaussianNB())
scores_partie_2 = ["roc_auc", "recall"]
pd.concat(
  {
    "Régression logistique": evaluer(modele, scores_partie_2),
    "Bayésien naïf": evaluer(bayes, scores_partie_2),
  },
  axis=1,
).round(3)

Le bayésien naïf repère bien plus de départs (un rappel de 84 % contre 55 %), mais son AUC est plus basse (0,820 contre 0,845) : il classe moins bien les clients, du plus risqué au moins risqué. Son rappel vient de ce qu'il annonce beaucoup de départs, dont plus de la moitié à tort (une précision de 46 %) : un rappel ne se lit qu'avec la précision, et tous deux dépendent du seuil de 0,5 que `predict` applique. L'AUC, elle, ne dépend d'aucun seuil.

Tout autre classifieur vu en cours (k plus proches voisins, forêt aléatoire…) se met à la place de `GaussianNB()` dans `make_pipeline`, sans rien changer d'autre : le prétraitement et `evaluer` restent les mêmes.

### 6. Les courbes ROC

*Entraînez `bayes` sur l'entraînement, puis tracez sur les mêmes axes les courbes ROC des deux modèles sur le test avec `RocCurveDisplay.from_estimator`, et affichez l'AUC de chacun avec `roc_auc_score`. Lequel garder ? Pourquoi l'AUC ne dépend-elle pas du seuil ?*

**Documentation utile**

- [`sklearn.metrics.RocCurveDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html#sklearn.metrics.RocCurveDisplay.from_estimator)
- [`sklearn.metrics.roc_auc_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `fig, ax = plt.subplots(figsize=(6, 6))`, puis `RocCurveDisplay.from_estimator(modele, X_test, y_test, name="Régression logistique", ax=ax)` : `ax=ax` trace sur ces axes, `name` légende la courbe. Même appel pour `bayes` ; `plot_chance_level=True` sur le second ajoute la diagonale du hasard.
- `roc_auc_score(y_test, probas)` prend les probabilités de départ, pas les classes prédites : `classifieur.predict_proba(X_test)[:, 1]`, la seconde colonne étant la probabilité de la classe 1.
- AUC attendues : environ 0,84 pour la régression logistique et 0,81 pour le bayésien naïf.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
bayes.fit(..., ...)

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_estimator(
  modele, X_test, y_test, name="Régression logistique", ax=ax
)
RocCurveDisplay.from_estimator(
  bayes, ..., ..., name="Bayésien naïf", ax=..., plot_chance_level=True
)
plt.show()

for nom, classifieur in [("Régression logistique", modele), ("Bayésien naïf", bayes)]:
  auc = roc_auc_score(y_test, classifieur.predict_proba(X_test)[...])
  print(f"AUC, {nom} : {auc:.3f}")
```

#### Solution

In [ ]:
bayes.fit(X_entrainement, y_entrainement)

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_estimator(
  modele, X_test, y_test, name="Régression logistique", ax=ax
)
RocCurveDisplay.from_estimator(
  bayes, X_test, y_test, name="Bayésien naïf", ax=ax, plot_chance_level=True
)
plt.show()

for nom, classifieur in [("Régression logistique", modele), ("Bayésien naïf", bayes)]:
  auc = roc_auc_score(y_test, classifieur.predict_proba(X_test)[:, 1])
  print(f"AUC, {nom} : {auc:.3f}")

La courbe de la régression logistique passe au-dessus de celle du bayésien naïf sur presque toute sa longueur, et son AUC est plus haute sur le test aussi (0,844 contre 0,809) : on garde la régression logistique.

Chaque point d'une courbe est un seuil : en le baissant, on repère plus de départs (on monte) au prix de plus de fausses alertes (on va à droite). L'AUC résume la courbe entière, donc tous les seuils à la fois : elle mesure comment le modèle ordonne les clients (c'est la probabilité qu'un client parti tiré au hasard ait une probabilité de départ plus haute qu'un client resté tiré au hasard), pas la décision qu'on en tire. Reste à choisir ce seuil.

### 7. Le seuil par le coût des erreurs

Le seuil de 0,5 n'a rien de particulier : c'est le coût des erreurs qui le fixe. Supposons, pour l'opérateur (ces chiffres sont des hypothèses, pas des données) :

- un client qui part sans avoir reçu d'offre fait perdre 500 $ de chiffre d'affaires (environ 7 mois de facture d'un client qui part) ;
- une offre de fidélisation coûte 100 $ à chaque client qui la reçoit, qu'il serait parti ou non, et retient à coup sûr un client qui allait partir.

Le coût total d'un seuil est donc de 500 $ par client parti sans offre (un faux négatif) et de 100 $ par offre (un vrai ou un faux positif).

Le seuil se choisit sur l'entraînement, sans toucher au test. Il faut pour cela des probabilités prédites pour des clients que le modèle n'a pas vus : `cross_val_predict` les donne, chaque client recevant la probabilité prédite par le modèle du pli où il servait à la validation.

*Écrivez la fonction `cout_total(y_vrai, proba, seuil)`, qui renvoie le coût total d'une offre faite à chaque client dont la probabilité de départ `proba` atteint `seuil`. Calculez ensuite les probabilités hors échantillon `proba_entrainement` de `modele` sur l'entraînement avec `cross_val_predict` et `plis`, tracez le coût total pour les seuils de 0,05 à 0,95 par pas de 0,05, et gardez dans `seuil_optimal` le seuil qui le minimise. La section qui suit l'applique au test.*

**Documentation utile**

- [`sklearn.model_selection.cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)

In [ ]:
COUT_DEPART = 500  # $ : un client parti sans avoir reçu d'offre
COUT_OFFRE = 100  # $ : une offre de fidélisation


def cout_total(y_vrai, proba, seuil): ...


# Votre code ici

#### Conseils supplémentaires

- Dans `cout_total` : `offre = proba >= seuil` donne un tableau de booléens, vrai pour chaque client qui reçoit une offre. Les départs manqués sont `(np.asarray(y_vrai) == 1) & ~offre` (`~` : la négation), et `.sum()` compte les vrais d'un tableau de booléens. Un petit exemple à vérifier : `cout_total(np.array([1, 0, 1]), np.array([0.9, 0.6, 0.2]), 0.5)` vaut 700 (deux offres et un départ manqué).
- `cross_val_predict(modele, X_entrainement, y_entrainement, cv=plis, method="predict_proba")` renvoie deux colonnes, comme `predict_proba` : `[:, 1]` garde la probabilité de départ.
- `np.linspace(0.05, 0.95, 19)` : les 19 seuils de 0,05 à 0,95 ; `np.argmin(couts)` : la position du plus petit coût.
- Seuil optimal attendu : 0,15, avec un coût presque aussi bas de 0,15 à 0,25.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def cout_total(y_vrai, proba, seuil):
  offre = proba >= seuil
  departs_manques = (np.asarray(y_vrai) == 1) & ~offre
  return COUT_DEPART * ... + COUT_OFFRE * ...


proba_entrainement = cross_val_predict(
  modele, X_entrainement, y_entrainement, cv=plis, method=...
)[:, 1]
seuils = np.linspace(0.05, 0.95, 19)
couts = [cout_total(..., ..., seuil) for seuil in seuils]
seuil_optimal = seuils[...]

fig, ax = plt.subplots()
ax.plot(seuils, couts, marker="o")
ax.axvline(seuil_optimal, color="tab:red", linestyle="--")
ax.set_xlabel("Seuil")
ax.set_ylabel("Coût total sur l'entraînement ($)")
plt.show()
print(f"Seuil optimal : {seuil_optimal:.2f}")
```

#### Solution

In [ ]:
def cout_total(y_vrai, proba, seuil):
  offre = proba >= seuil
  departs_manques = (np.asarray(y_vrai) == 1) & ~offre
  return COUT_DEPART * departs_manques.sum() + COUT_OFFRE * offre.sum()


proba_entrainement = cross_val_predict(
  modele, X_entrainement, y_entrainement, cv=plis, method="predict_proba"
)[:, 1]
seuils = np.linspace(0.05, 0.95, 19)
couts = [cout_total(y_entrainement, proba_entrainement, seuil) for seuil in seuils]
seuil_optimal = seuils[np.argmin(couts)]

fig, ax = plt.subplots()
ax.plot(seuils, couts, marker="o")
ax.axvline(seuil_optimal, color="tab:red", linestyle="--")
ax.set_xlabel("Seuil")
ax.set_ylabel("Coût total sur l'entraînement ($)")
plt.show()
print(f"Seuil optimal : {seuil_optimal:.2f}")

### Le seuil choisi, sur le test

`seuil_optimal`, appliqué au test, face au seuil de 0,5 et à deux repères : ne faire aucune offre, ou en faire une à tous les clients.

In [ ]:
proba_test = modele.predict_proba(X_test)[:, 1]
print(f"Aucune offre : {COUT_DEPART * y_test.sum()} $")
print(f"Une offre à tous les clients : {COUT_OFFRE * len(y_test)} $")
pd.DataFrame(
  [
    {
      "Seuil": seuil,
      "Offres": (proba_test >= seuil).sum(),
      "Coût total ($)": cout_total(y_test, proba_test, seuil),
      "Précision": precision_score(y_test, proba_test >= seuil),
      "Rappel": recall_score(y_test, proba_test >= seuil),
    }
    for seuil in [0.5, seuil_optimal]
  ]
).round(2)

Sur le test, le seuil de 0,15 coûte 114 100 $, contre 149 100 $ au seuil de 0,5 : 35 000 $ d'économie, et bien moins que de ne faire aucune offre (233 500 $) ou d'en faire une à tous (176 100 $). Le modèle fait alors 956 offres au lieu de 396 : sa précision tombe de 63 % à 45 %, mais son rappel monte de 53 % à 92 %. Une offre coûte cinq fois moins cher qu'un départ manqué : mieux vaut beaucoup de fausses alertes qu'un départ de plus.

Ce seuil dépend des hypothèses : avec d'autres coûts, refaites le calcul. D'ailleurs, une offre est rentable dès que la probabilité de départ dépasse 100 / 500 = 0,2 : si les probabilités du modèle sont fiables, le seuil optimal en est proche, et la courbe est presque plate autour. scikit-learn (1.5 et suivants) fait ce choix en une classe, [`TunedThresholdClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TunedThresholdClassifierCV.html), qui règle le seuil par validation croisée d'après un score donné, un coût métier compris.

## Conclusion

- Sur des classes déséquilibrées, l'exactitude trompe : une référence qui ne repère aucun départ obtient 73,5 %. Regardez la matrice de confusion, la précision et le rappel de la classe qui compte.
- Pour comparer des modèles, la validation croisée sur l'entraînement : une moyenne, et un écart type qui dit ce que vaut une différence. Le test reste pour la mesure finale.
- L'AUC classe les modèles sans dépendre du seuil : elle mesure comment ils ordonnent les clients (partie 2).
- Le seuil, lui, se décide d'après le coût des erreurs pour le métier, sur des probabilités hors échantillon de l'entraînement : ici 0,15 plutôt que 0,5 (partie 2).